In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install lightgbm wandb transformers --quiet
!pip install -U bitsandbytes>=0.46.1 --quiet
!pip install -q peft trl datasets bitsandbytes accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# ── W&B Setup — bulletproof for Kaggle kernels ──
import wandb

_wandb_ready = False
try:
    from kaggle_secrets import UserSecretsClient
    _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    wandb.login(key=_key, relogin=True)
    _wandb_ready = True
    print("W&B: logged in via Kaggle Secrets")
except Exception as e:
    print(f"W&B: Could not login ({e}), switching to offline mode")
    os.environ["WANDB_MODE"] = "offline"
    _wandb_ready = True  # offline mode still works

def safe_wandb_init(**kwargs):
    """Wrapper that never crashes — falls back to offline mode."""
    try:
        return wandb.init(**kwargs)
    except Exception:
        os.environ["WANDB_MODE"] = "offline"
        return wandb.init(**kwargs)

print(f"W&B mode: {os.environ.get('WANDB_MODE', 'online')}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B: logged in via Kaggle Secrets
W&B mode: online


In [4]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 1: Enhanced TF-IDF + LightGBM
============================================================
Classical ML Baseline (Milestone 2 requirement)
Metric: MAP@3 (Mean Average Precision at 3)

Enhanced with: TF-IDF SVD dense embeddings, Jaccard overlap,
character n-gram overlap, option-vs-option pairwise similarity,
and comprehensive text statistics.
============================================================
"""

import pandas as pd
import numpy as np
import os, re, warnings
from itertools import combinations
warnings.filterwarnings("ignore")

import wandb
# API key already set as env var in setup cell

# ── MAP@3 Evaluation ──
def map_at_3(y_true, y_pred_top3):
    """Compute Mean Average Precision @ 3."""
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data Loading ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Answer distribution:\n{train['answer'].value_counts()}")

LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Advanced Feature Engineering ──
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix

def tokenize_words(text):
    return set(re.findall(r'\w+', str(text).lower()))

def jaccard_sim(t1, t2):
    s1, s2 = tokenize_words(t1), tokenize_words(t2)
    if not s1 or not s2: return 0.0
    return len(s1 & s2) / len(s1 | s2)

def common_word_count(t1, t2):
    s1, s2 = tokenize_words(t1), tokenize_words(t2)
    return len(s1 & s2)

def char_ngram_overlap(t1, t2, n=3):
    def ngrams(t):
        t = str(t).lower()
        return set(t[i:i+n] for i in range(len(t)-n+1))
    ng1, ng2 = ngrams(t1), ngrams(t2)
    if not ng1 or not ng2: return 0.0
    return len(ng1 & ng2) / max(len(ng1 | ng2), 1)

def build_features(df, shared_vec=None, svd_model=None, fit=True):
    """Build comprehensive features for LightGBM."""
    prompts = df['prompt'].fillna('').values
    opts = {c: df[c].fillna('').values for c in LABELS}
    N = len(df)

    # ── TF-IDF + SVD ──
    all_text = list(prompts)
    for c in LABELS:
        all_text.extend(list(opts[c]))

    if fit:
        shared_vec = TfidfVectorizer(max_features=15000, ngram_range=(1,2),
                                      sublinear_tf=True, min_df=2)
        shared_vec.fit(all_text)
        svd_model = TruncatedSVD(n_components=128, random_state=42)
        svd_model.fit(shared_vec.transform(all_text))

    prompt_tfidf = shared_vec.transform(prompts)
    opt_tfidf = {c: shared_vec.transform(opts[c]) for c in LABELS}
    prompt_svd = svd_model.transform(prompt_tfidf)
    opt_svd = {c: svd_model.transform(opt_tfidf[c]) for c in LABELS}

    feats = []

    # 1. TF-IDF cosine sim: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array(prompt_tfidf.multiply(opt_tfidf[c]).sum(axis=1)).flatten())

    # 2. SVD cosine sim: prompt ↔ each option
    for c in LABELS:
        norms_p = np.linalg.norm(prompt_svd, axis=1, keepdims=True) + 1e-8
        norms_o = np.linalg.norm(opt_svd[c], axis=1, keepdims=True) + 1e-8
        feats.append(np.sum(prompt_svd * opt_svd[c], axis=1) / (norms_p.flatten() * norms_o.flatten()))

    # 3. Jaccard sim: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array([jaccard_sim(prompts[i], opts[c][i]) for i in range(N)]))

    # 4. Common word count: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array([common_word_count(prompts[i], opts[c][i]) for i in range(N)]))

    # 5. Char n-gram overlap: prompt ↔ each option
    for c in LABELS:
        feats.append(np.array([char_ngram_overlap(prompts[i], opts[c][i]) for i in range(N)]))

    # 6. Option-vs-option Jaccard (pairwise) — 10 pairs
    for c1, c2 in combinations(LABELS, 2):
        feats.append(np.array([jaccard_sim(opts[c1][i], opts[c2][i]) for i in range(N)]))

    # 7. Text length features
    feats.append(np.array([len(str(p)) for p in prompts]))  # prompt char len
    feats.append(np.array([len(str(p).split()) for p in prompts]))  # prompt word count
    for c in LABELS:
        feats.append(np.array([len(str(o)) for o in opts[c]]))  # option char len
        feats.append(np.array([len(str(o).split()) for o in opts[c]]))  # option word count

    # 8. Statistical features across options
    opt_lens = np.column_stack([[len(str(o)) for o in opts[c]] for c in LABELS])
    feats.append(opt_lens.mean(axis=1))   # mean option length
    feats.append(opt_lens.std(axis=1))    # std option length
    feats.append(opt_lens.max(axis=1) - opt_lens.min(axis=1))  # range

    # 9. Relative length features (each option length / mean option length)
    mean_lens = opt_lens.mean(axis=1, keepdims=True) + 1e-8
    for i, c in enumerate(LABELS):
        feats.append(opt_lens[:, i] / mean_lens.flatten())

    # Combine sparse TF-IDF + dense features
    sparse_feats = hstack([prompt_tfidf] + [opt_tfidf[c] for c in LABELS])
    dense_feats = np.column_stack(feats)
    combined = hstack([sparse_feats, csr_matrix(dense_feats)])

    return combined, shared_vec, svd_model

print("Building features...")
X_train, shared_vec, svd_model = build_features(train, fit=True)
X_test, _, _ = build_features(test, shared_vec, svd_model, fit=False)
y_train = train['answer'].map(label2id).values
print(f"Features: X_train={X_train.shape}, X_test={X_test.shape}")

# ── Cross-Validation with LightGBM ──
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))

run = safe_wandb_init(
    project="smart-mcq-solver",
    name="model1-enhanced-lgbm",
    config={"model": "LightGBM", "features": "TF-IDF+SVD+Overlap", "n_folds": N_FOLDS}
)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1}/{N_FOLDS} ---")
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=95,
        max_depth=10, subsample=0.75, colsample_bytree=0.4,
        reg_alpha=0.3, reg_lambda=2.0, min_child_samples=15,
        random_state=42+fold, n_jobs=-1, verbose=-1
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
    )

    val_probs = model.predict_proba(X_val)
    oof_probs[val_idx] = val_probs
    test_probs += model.predict_proba(X_test) / N_FOLDS

    val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_probs]
    val_true = [id2label[y] for y in y_val]
    fold_map3 = map_at_3(val_true, val_top3)
    print(f"Fold {fold+1} MAP@3: {fold_map3:.4f}")
    wandb.log({"fold": fold+1, "fold_map3": fold_map3})

oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_train]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 1 (Enhanced LightGBM) OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("lgbm_test_probs.npy", test_probs)
np.save("lgbm_oof_probs.npy", oof_probs)

test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_lgbm.csv", index=False)
print("Model 1 submission saved to submission_lgbm.csv")
print(sub.head(10))


Train: (2000, 8), Test: (500, 7)
Answer distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Building features...
Features: X_train=(2000, 74851), X_test=(500, 74851)


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260608_082746-objhgtu6
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-enhanced-lgbm
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/objhgtu6



--- Fold 1/5 ---
Training until validation scores don't improve for 100 rounds
[200]	valid_0's multi_logloss: 0.0199599
[400]	valid_0's multi_logloss: 0.0100209
[600]	valid_0's multi_logloss: 0.00958654
[800]	valid_0's multi_logloss: 0.00958606
[1000]	valid_0's multi_logloss: 0.00958573
[1200]	valid_0's multi_logloss: 0.00958546
[1400]	valid_0's multi_logloss: 0.00958527
[1600]	valid_0's multi_logloss: 0.00958517
[1800]	valid_0's multi_logloss: 0.00958508
[2000]	valid_0's multi_logloss: 0.00958503
Did not meet early stopping. Best iteration is:
[1995]	valid_0's multi_logloss: 0.00958503
Fold 1 MAP@3: 1.0000

--- Fold 2/5 ---
Training until validation scores don't improve for 100 rounds
[200]	valid_0's multi_logloss: 0.0186434
[400]	valid_0's multi_logloss: 0.00933168
[600]	valid_0's multi_logloss: 0.00892795
[800]	valid_0's multi_logloss: 0.00890391
Early stopping, best iteration is:
[767]	valid_0's multi_logloss: 0.00890391
Fold 2 MAP@3: 1.0000

--- Fold 3/5 ---
Training until valida

wandb: updating run metadata


Fold 5 MAP@3: 1.0000

=== Model 1 (Enhanced LightGBM) OOF MAP@3: 1.0000 ===


wandb: uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:         fold ▁▃▅▆█
wandb:    fold_map3 ▁▁▁▁▁
wandb: overall_map3 ▁
wandb: 
wandb: Run summary:
wandb:         fold 5
wandb:    fold_map3 1
wandb: overall_map3 1
wandb: 
wandb: 🚀 View run model1-enhanced-lgbm at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/objhgtu6
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260608_082746-objhgtu6/logs


Model 1 submission saved to submission_lgbm.csv
   ID Prediction
0   1      A C D
1   2      B D C
2   3      B E A
3   4      E C A
4   5      C D E
5   6      D C A
6   7      E D B
7   8      B D C
8   9      C D A
9  10      B C D


In [5]:
"""
============================================================
SMART MCQ SOLVER CHALLENGE — MODEL 2: PyTorch Text CNN (From Scratch)
============================================================
Neural Network built FROM SCRATCH (Milestone 3 requirement)
Uses pre-trained GloVe embeddings + custom CNN with attention.

Improvements over baseline:
- Longer sequence lengths (128 tokens)
- Multi-scale CNN with attention pooling
- Cross-attention between prompt and option
- Label smoothing, OneCycleLR, early stopping
- Proper dropout and batch normalization
============================================================
"""

import pandas as pd
import numpy as np
import os, gc, re, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import wandb

# ── Reproducibility ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── MAP@3 ──
def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

# ── Data ──
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A', 'B', 'C', 'D', 'E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

# ── Tokenizer ──
from collections import Counter

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

# Build vocabulary from train + test
all_texts_flat = list(train['prompt']) + list(test['prompt'])
for c in LABELS:
    all_texts_flat += list(train[c]) + list(test[c])

word_counts = Counter()
for text in all_texts_flat:
    word_counts.update(tokenize(str(text)))

VOCAB_SIZE = 60000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in word_counts.most_common(VOCAB_SIZE):
    vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)}")

def encode_text(text, max_len=128):
    tokens = tokenize(str(text))[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids = ids + [0] * (max_len - len(ids))
    return ids

# ── Load GloVe Embeddings ──
GLOVE_DIM = 100
GLOVE_PATH = "/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.100d.txt"
if not os.path.exists(GLOVE_PATH):
    GLOVE_PATH = "glove.6B.100d.txt"

embedding_matrix = np.random.normal(0, 0.05, (len(vocab), GLOVE_DIM)).astype(np.float32)
embedding_matrix[0] = 0  # PAD = zeros

if os.path.exists(GLOVE_PATH):
    print("Loading GloVe embeddings...")
    loaded = 0
    with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.array(parts[1:], dtype=np.float32)
                loaded += 1
    print(f"GloVe loaded: {loaded}/{len(vocab)} words covered")
else:
    print("WARNING: GloVe not found, using random embeddings.")
    print("Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.")

# ── Dataset ──
MAX_PROMPT_LEN = 128
MAX_OPTION_LEN = 128

class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = encode_text(row['prompt'], MAX_PROMPT_LEN)
        option_ids = [encode_text(row[c], MAX_OPTION_LEN) for c in LABELS]

        prompt_t = torch.tensor(prompt_ids, dtype=torch.long)
        options_t = torch.tensor(option_ids, dtype=torch.long)

        if self.is_test:
            return prompt_t, options_t
        else:
            label = label2id[row['answer']]
            return prompt_t, options_t, torch.tensor(label, dtype=torch.long)

# ── Improved Text CNN Model (from scratch, with attention) ──
class AttentionPool(nn.Module):
    """Self-attention pooling layer."""
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x):
        # x: (B, L, D)
        w = torch.softmax(self.attn(x), dim=1)  # (B, L, 1)
        return (x * w).sum(dim=1)  # (B, D)

class TextCNNMCQ(nn.Module):
    """
    Custom Text CNN for Multiple Choice QA (built from scratch).
    Architecture:
    - GloVe embeddings + learnable projection
    - Multi-kernel 1D CNN (sizes 2,3,4,5,7)
    - Attention pooling over CNN features
    - Cross-attention between prompt and option
    - FC classifier with residual connection
    """
    def __init__(self, pretrained_embeddings, kernel_sizes=[2, 3, 4, 5, 7],
                 num_filters=192, hidden_dim=384, dropout=0.35):
        super().__init__()
        vocab_size, embed_dim = pretrained_embeddings.shape

        # Embedding (frozen GloVe + trainable projection)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings), requires_grad=False
        )
        self.embed_proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        # Multi-kernel CNN — padding='same' ensures all kernels output same length
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k, padding='same')
            for k in kernel_sizes
        ])
        self.conv_bn = nn.ModuleList([
            nn.BatchNorm1d(num_filters) for _ in kernel_sizes
        ])

        cnn_out = num_filters * len(kernel_sizes)

        # Attention pooling
        self.attn_pool = AttentionPool(cnn_out)

        # Cross-attention: prompt attends to option
        self.cross_attn = nn.MultiheadAttention(cnn_out, num_heads=4, dropout=0.1, batch_first=True)
        self.cross_norm = nn.LayerNorm(cnn_out)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(cnn_out * 3, hidden_dim),  # prompt + option + cross-attn
            nn.GELU(),
            nn.Dropout(dropout),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1)
        )

    def encode_text_seq(self, x):
        """x: (B, L) → (B, L', cnn_out) sequence of features"""
        emb = self.embedding(x)
        emb = self.embed_proj(emb)
        emb = emb.permute(0, 2, 1)  # (B, D, L)

        conv_outs = []
        for conv, bn in zip(self.convs, self.conv_bn):
            c = F.gelu(bn(conv(emb)))  # (B, F, L')
            conv_outs.append(c)

        # Concatenate along filter dim
        out = torch.cat(conv_outs, dim=1)  # (B, cnn_out, L')
        return out.permute(0, 2, 1)  # (B, L', cnn_out)

    def encode_text(self, x):
        """x: (B, L) → (B, cnn_out) pooled features"""
        seq = self.encode_text_seq(x)
        return self.attn_pool(seq)

    def forward(self, prompt, options):
        """
        prompt:  (B, prompt_len)
        options: (B, 5, option_len)
        returns: (B, 5) logits
        """
        B = prompt.size(0)
        prompt_seq = self.encode_text_seq(prompt)   # (B, L, D)
        prompt_feat = self.attn_pool(prompt_seq)     # (B, D)

        logits = []
        for i in range(5):
            opt_seq = self.encode_text_seq(options[:, i, :])  # (B, L, D)
            opt_feat = self.attn_pool(opt_seq)                 # (B, D)

            # Cross-attention: prompt attends to option
            cross_out, _ = self.cross_attn(
                prompt_seq, opt_seq, opt_seq
            )
            cross_feat = self.attn_pool(self.cross_norm(cross_out))  # (B, D)

            combined = torch.cat([prompt_feat, opt_feat, cross_feat], dim=1)
            score = self.fc(combined)
            logits.append(score)

        return torch.cat(logits, dim=1)

# ── Training ──
N_FOLDS = 5
EPOCHS = 20
BATCH_SIZE = 32
LR = 8e-4

# API key already set as env var in setup cell

safe_wandb_init(project="smart-mcq-solver", name="model2-textcnn-attn",
           config={"model": "TextCNN+Attention", "epochs": EPOCHS, "lr": LR,
                   "batch_size": BATCH_SIZE, "filters": 192, "kernels": "2,3,4,5,7"})

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
y_all = train['answer'].map(label2id).values
oof_probs = np.zeros((len(train), 5))
test_probs = np.zeros((len(test), 5))
test_ds = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Label smoothing cross-entropy
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_preds = F.log_softmax(pred, dim=-1)
        nll = -log_preds.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_preds.mean(dim=-1)
        loss = (1 - self.smoothing) * nll + self.smoothing * smooth_loss
        return loss.mean()

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y_all)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    train_ds = MCQDataset(train.iloc[tr_idx])
    val_ds   = MCQDataset(train.iloc[val_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = TextCNNMCQ(embedding_matrix).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=1e-3
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS,
        steps_per_epoch=len(train_loader), pct_start=0.1
    )
    criterion = LabelSmoothingCE(smoothing=0.1)

    best_map3 = 0
    patience = 5
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for prompt, options, labels in train_loader:
            prompt, options, labels = prompt.to(DEVICE), options.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(prompt, options)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds_all, val_labels_all = [], []
        with torch.no_grad():
            for prompt, options, labels in val_loader:
                prompt, options = prompt.to(DEVICE), options.to(DEVICE)
                logits = model(prompt, options)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                val_preds_all.append(probs)
                val_labels_all.append(labels.numpy())

        val_preds_all = np.concatenate(val_preds_all)
        val_labels_all = np.concatenate(val_labels_all)

        val_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in val_preds_all]
        val_true = [id2label[y] for y in val_labels_all]
        epoch_map3 = map_at_3(val_true, val_top3)

        avg_loss = train_loss / len(train_loader)
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val MAP@3: {epoch_map3:.4f}")
        wandb.log({"fold": fold+1, "epoch": epoch+1, "train_loss": avg_loss, "val_map3": epoch_map3})

        if epoch_map3 > best_map3:
            best_map3 = epoch_map3
            best_val_preds = val_preds_all.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"Fold {fold+1} Best MAP@3: {best_map3:.4f}")
    oof_probs[val_idx] = best_val_preds

    # Test inference with best model
    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    fold_test_preds = []
    with torch.no_grad():
        for prompt, options in test_loader:
            prompt, options = prompt.to(DEVICE), options.to(DEVICE)
            logits = model(prompt, options)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            fold_test_preds.append(probs)
    test_probs += np.concatenate(fold_test_preds) / N_FOLDS

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

# Overall OOF MAP@3
oof_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in oof_probs]
oof_true = [id2label[y] for y in y_all]
overall_map3 = map_at_3(oof_true, oof_top3)
print(f"\n=== Model 2 (TextCNN+Attn) Overall OOF MAP@3: {overall_map3:.4f} ===")
wandb.log({"overall_map3": overall_map3})
wandb.finish()

np.save("textcnn_test_probs.npy", test_probs)
np.save("textcnn_oof_probs.npy", oof_probs)

test_top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in test_probs]
sub = pd.DataFrame({
    "ID": test["id"],
    "Prediction": [" ".join(t) for t in test_top3]
})
sub.to_csv("submission_textcnn.csv", index=False)
print("Model 2 submission saved to submission_textcnn.csv")
print(sub.head(10))


Device: cuda
Vocabulary size: 2975
Add 'glove-global-vectors-for-word-representation' as a Kaggle dataset.


wandb: setting up run k8vydhpe
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260608_082944-k8vydhpe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-textcnn-attn
wandb: ⭐️ View project at https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/k8vydhpe



Fold 1/5
Epoch 1/20 | Loss: 1.5932 | Val MAP@3: 0.6062
Epoch 2/20 | Loss: 1.1542 | Val MAP@3: 0.8554
Epoch 4/20 | Loss: 0.6419 | Val MAP@3: 0.9738
Epoch 6/20 | Loss: 0.5175 | Val MAP@3: 0.9875
Epoch 8/20 | Loss: 0.4624 | Val MAP@3: 0.9762
Epoch 10/20 | Loss: 0.4310 | Val MAP@3: 0.9938
Epoch 12/20 | Loss: 0.4189 | Val MAP@3: 0.9838
Epoch 14/20 | Loss: 0.4135 | Val MAP@3: 0.9938
  Early stopping at epoch 15
Fold 1 Best MAP@3: 0.9938

Fold 2/5
Epoch 1/20 | Loss: 1.6072 | Val MAP@3: 0.5642
Epoch 2/20 | Loss: 1.2148 | Val MAP@3: 0.8842
Epoch 4/20 | Loss: 0.6601 | Val MAP@3: 0.9571
Epoch 6/20 | Loss: 0.5367 | Val MAP@3: 0.9896
Epoch 8/20 | Loss: 0.4767 | Val MAP@3: 0.9962
Epoch 10/20 | Loss: 0.4303 | Val MAP@3: 1.0000
Epoch 12/20 | Loss: 0.4142 | Val MAP@3: 1.0000
Epoch 14/20 | Loss: 0.4057 | Val MAP@3: 0.9862
  Early stopping at epoch 15
Fold 2 Best MAP@3: 1.0000

Fold 3/5
Epoch 1/20 | Loss: 1.5947 | Val MAP@3: 0.5988
Epoch 2/20 | Loss: 1.1680 | Val MAP@3: 0.8517
Epoch 4/20 | Loss: 0.6528 

wandb: updating run metadata



=== Model 2 (TextCNN+Attn) Overall OOF MAP@3: 0.9954 ===


wandb: uploading history steps 72-73, summary, console lines 68-71
wandb: 
wandb: Run history:
wandb:        epoch ▁▂▃▃▅▆▆▇▇█▂▂▃▄▅▆▇█▃▃▅▅▆▆▇▂▂▄▅▆▇██▁▁▃▄▄▅▇
wandb:         fold ▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆████████
wandb: overall_map3 ▁
wandb:   train_loss █▃▂▂▂▁▁▁▁▁█▁▁▁▁▁█▅▃▂▁▁▁▁█▃▂▂▁▁▁▁▁▁▁▃▂▂▁▁
wandb:     val_map3 ▂▆▇██████▁██▇████▂▆████▆▇███████▂▇▇█████
wandb: 
wandb: Run summary:
wandb:        epoch 13
wandb:         fold 5
wandb: overall_map3 0.99542
wandb:   train_loss 0.425
wandb:     val_map3 0.98625
wandb: 
wandb: 🚀 View run model2-textcnn-attn at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/k8vydhpe
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260608_082944-k8vydhpe/logs


Model 2 submission saved to submission_textcnn.csv
   ID Prediction
0   1      A B E
1   2      B E D
2   3      B D E
3   4      E C D
4   5      C D B
5   6      D C A
6   7      E C A
7   8      B E A
8   9      C D A
9  10      B D C


In [6]:
"""
============================================================
MODEL 3: QLoRA Fine-Tuned Qwen2.5-7B-Instruct (Optimized)
============================================================
Pre-trained Transformer model (Milestone 5 requirement)
Fine-tuned with QLoRA on MCQ answering task.

Optimizations over v7:
- LoRA on 7 target modules (was 2): q/k/v/o_proj, gate/up/down_proj
- r=32 with lora_alpha=64 (was r=16, alpha=32)
- 2 training epochs (was 1) — fixes underfitting
- max_length=512 (was 384) — captures full context
- learning_rate=2e-4 with cosine schedule
- gradient_accumulation_steps=4 for effective batch=16
- NEFTune noise for better generalization
- Improved probability extraction from logits
============================================================
"""
import pandas as pd, numpy as np, os, gc, re, warnings, torch
warnings.filterwarnings("ignore")
import wandb
# API key already set as env var in setup cell

LABELS = ['A','B','C','D','E']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i:l for i,l in enumerate(LABELS)}
def map_at_3(y_true, y_pred_top3):
    return np.mean([1.0/(next((i+1 for i,p in enumerate(pr[:3]) if p==t),4))
                     for t,pr in zip(y_true,y_pred_top3)])

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR): DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

# ═══ LOAD MODEL ═══
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Try multiple model paths (Kaggle dataset, then HF hub)
model_path = None
for p in ["/kaggle/input/qwen2.5/transformers/7b-instruct/1",
          "/kaggle/input/qwen-2-5/transformers/7b-instruct/1",
          "Qwen/Qwen2.5-7B-Instruct"]:
    if not p.startswith("/") or os.path.exists(p):
        model_path = p
        break
print(f"Loading: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_path,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    ),
    device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
print("✓ Model loaded")

# ═══ QLORA SETUP — Aggressive fine-tuning ═══
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from datasets import Dataset

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA on ALL attention + MLP modules for maximum adaptation
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ═══ Format training data — improved prompt ═══
def format_mcq(row):
    msgs = [
        {"role": "system", "content": "You are an expert at answering multiple choice questions. Read the question and all options carefully, then select the single best answer."},
        {"role": "user", "content":
         f"Question: {row['prompt']}\n\n"
         f"A) {row['A']}\n"
         f"B) {row['B']}\n"
         f"C) {row['C']}\n"
         f"D) {row['D']}\n"
         f"E) {row['E']}\n\n"
         f"Answer with ONLY the correct letter (A, B, C, D, or E):"},
        {"role": "assistant", "content": str(row['answer'])}
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

texts = [format_mcq(row) for _, row in train.iterrows()]
print(f"Training: {len(texts)} samples")

def tok_fn(ex):
    e = tokenizer(ex["text"], truncation=True, max_length=512, padding="max_length")
    e["labels"] = e["input_ids"].copy()
    return e

ds = Dataset.from_dict({"text": texts}).map(tok_fn, remove_columns=["text"])
ds.set_format("torch")

# ═══ TRAIN — 2 epochs, larger effective batch ═══
from transformers import Trainer, TrainingArguments

safe_wandb_init(project="smart-mcq-solver", name="model3-qlora-optimized",
           config={"model": model_path, "r": 32, "alpha": 64,
                   "targets": "q,k,v,o,gate,up,down", "epochs": 2,
                   "lr": 2e-4, "eff_batch": 16})

training_args = TrainingArguments(
    output_dir="/kaggle/working/qlora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=20,
    save_strategy="no",
    report_to="wandb",
    remove_unused_columns=False,
    neftune_noise_alpha=5,   # NEFTune: adds noise to embeddings for better generalization
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
)

trainer = Trainer(model=model, train_dataset=ds, args=training_args)
trainer.train()
model = model.merge_and_unload()
model.eval()
print("✓ Fine-tuning done (2 epochs, 7 LoRA targets, r=32)")

# ═══ INFERENCE — improved probability extraction ═══
# Pre-compute token IDs for each answer letter
tok_ids = {}
for l in LABELS:
    ids = set()
    for fmt in [f" {l}", l, f" {l})", f"{l})", f" {l}."]:
        encoded = tokenizer.encode(fmt, add_special_tokens=False)
        if encoded:
            ids.add(encoded[-1])
            if len(encoded) > 1:
                ids.add(encoded[0])
    tok_ids[l] = list(ids)

def parse_letter(t):
    t = t.strip()
    m = re.match(r'^([A-E])', t)
    if m: return m.group(1)
    m = re.search(r'([A-E])', t)
    return m.group(1) if m else None

def answer_q(row):
    """Get answer probabilities for a single question."""
    msgs = [
        {"role": "system", "content": "You are an expert at answering multiple choice questions. Read the question and all options carefully, then select the single best answer."},
        {"role": "user", "content":
         f"Question: {row['prompt']}\n\n"
         f"A) {row['A']}\n"
         f"B) {row['B']}\n"
         f"C) {row['C']}\n"
         f"D) {row['D']}\n"
         f"E) {row['E']}\n\n"
         f"Answer with ONLY the correct letter (A, B, C, D, or E):"}
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048)
    inp = {k: v.to(model.device) for k, v in inp.items()}

    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=5, do_sample=False,
                             output_scores=True, return_dict_in_generate=True)

    gen = tokenizer.decode(out.sequences[0, inp['input_ids'].shape[1]:],
                           skip_special_tokens=True)
    ans = parse_letter(gen)

    # Extract probabilities from first token logits
    logits = out.scores[0][0].float()
    raw = np.array([max(logits[t].item() for t in tok_ids[l]) for l in LABELS])

    # Temperature-scaled softmax for calibrated probabilities
    temperature = 1.5
    probs = np.exp((raw - raw.max()) / temperature)
    probs /= probs.sum()

    # Boost the generated answer (if valid)
    if ans and ans in LABELS:
        probs[LABELS.index(ans)] += 0.3
        probs /= probs.sum()

    return probs

# ═══ Run inference on test set ═══
print(f"\n=== Test Inference ({len(test)} questions) ===")
tp = []
for i, row in test.iterrows():
    tp.append(answer_q(row))
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(test)} done")
tp = np.array(tp)
np.save("llm_test_probs.npy", tp)

# ═══ Also run inference on training set for ensemble OOF ═══
print(f"\n=== Train Inference for OOF ({len(train)} questions) ===")
train_probs = []
for i, row in train.iterrows():
    train_probs.append(answer_q(row))
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(train)} done")
train_probs = np.array(train_probs)
np.save("llm_oof_probs.npy", train_probs)

# Evaluate LLM OOF performance
llm_top3 = [[id2label[j] for j in np.argsort(-p)[:3]] for p in train_probs]
llm_true = list(train['answer'].values)
llm_map3 = map_at_3(llm_true, llm_top3)
print(f"\n=== Model 3 (QLoRA LLM) Train-set MAP@3: {llm_map3:.4f} ===")
print("(Note: this is on training data, so slightly optimistic)")
wandb.log({"llm_train_map3": llm_map3})

# Generate LLM-only submission
top3 = [[id2label[j] for j in np.argsort(-p)[:3]] for p in tp]
sub = pd.DataFrame({"ID": test["id"], "Prediction": [" ".join(t) for t in top3]})
sub.to_csv("submission_llm.csv", index=False)
print(f"✓ submission_llm.csv saved\n{sub.head(10)}")
wandb.finish()
del model; gc.collect(); torch.cuda.empty_cache()


Loading: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓ Fine-tuning done (2 epochs, 7 LoRA targets, r=32)

=== Test Inference (500 questions) ===
  50/500 done
  100/500 done
  150/500 done
  200/500 done
  250/500 done
  300/500 done
  350/500 done
  400/500 done
  450/500 done
  500/500 done

=== Train Inference for OOF (2000 questions) ===
  100/2000 done
  200/2000 done
  300/2000 done
  400/2000 done
  500/2000 done
  600/2000 done
  700/2000 done
  800/2000 done
  900/2000 done
  1000/2000 done
  1100/2000 done
  1200/2000 done
  1300/2000 done
  1400/2000 done
  1500/2000 done
  1600/2000 done
  1700/2000 done
  1800/2000 done
  1900/2000 done


wandb: updating run metadata


  2000/2000 done

=== Model 3 (QLoRA LLM) Train-set MAP@3: 0.8989 ===
(Note: this is on training data, so slightly optimistic)
✓ submission_llm.csv saved
   ID Prediction
0   1      A E D
1   2      B A D
2   3      C B D
3   4      E A C
4   5      C A D
5   6      A D C
6   7      A E D
7   8      B A D
8   9      A C B
9  10      B D E


wandb: uploading data
wandb: 
wandb: Run history:
wandb:      llm_train_map3 ▁
wandb:         train/epoch ▁▂▂▃▃▄▅▅▆▆▇██
wandb:   train/global_step ▁▂▂▃▃▄▅▅▆▆▇███
wandb:     train/grad_norm ▆▃▇█▆▄▂▁▁▂▁▁
wandb: train/learning_rate ▆██▇▆▅▅▄▃▂▁▁
wandb:          train/loss █▂▂▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:           llm_train_map3 0.89887
wandb:               total_flos 8.7875905388544e+16
wandb:              train/epoch 2
wandb:        train/global_step 250
wandb:          train/grad_norm 0.07755
wandb:      train/learning_rate 0.0
wandb:               train/loss 0.02165
wandb:               train_loss 0.29756
wandb:            train_runtime 7272.392
wandb: train_samples_per_second 0.55
wandb:                       +1 ...
wandb: 
wandb: 🚀 View run model3-qlora-optimized at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver/runs/i0fcj8la
wandb: ⭐️ View project at: https://wandb.ai/24f1002825-t22026/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact fil

In [7]:
"""
============================================================
SMART MCQ SOLVER — ENSEMBLE & FINAL SUBMISSION
============================================================
Combines all available model predictions using optimized
weight blending with temperature scaling.
============================================================
"""
import pandas as pd, numpy as np, os
from scipy.optimize import minimize

def map_at_3(y_true, y_pred_top3):
    scores = []
    for true, preds in zip(y_true, y_pred_top3):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/smart-mcq-solver-challenge"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
LABELS = ['A','B','C','D','E']
id2label = {i:l for i,l in enumerate(LABELS)}
y_all = train['answer'].values

# ── Load predictions ──
model_oof, model_test = {}, {}
for name, oof_f, test_f in [
    ("lgbm",    "lgbm_oof_probs.npy",    "lgbm_test_probs.npy"),
    ("textcnn", "textcnn_oof_probs.npy",  "textcnn_test_probs.npy"),
    ("llm",     "llm_oof_probs.npy",      "llm_test_probs.npy"),
]:
    if os.path.exists(test_f):
        model_test[name] = np.load(test_f)
        if os.path.exists(oof_f):
            model_oof[name] = np.load(oof_f)
            top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in model_oof[name]]
            sc = map_at_3(y_all, top3)
            print(f"  ✓ {name}: OOF MAP@3 = {sc:.4f}")
        else:
            print(f"  ✓ {name}: test preds loaded (no OOF)")

print(f"\nModels with OOF: {list(model_oof.keys())}")
print(f"Models with test: {list(model_test.keys())}")

# ── Temperature scaling per model ──
def apply_temperature(probs, temp=1.0):
    """Apply temperature scaling to probability matrix."""
    log_p = np.log(np.clip(probs, 1e-10, 1.0))
    scaled = np.exp(log_p / temp)
    return scaled / scaled.sum(axis=1, keepdims=True)

# ── Optimize ensemble weights using OOF predictions ──
if len(model_oof) > 1:
    names = list(model_oof.keys())
    oofs = [model_oof[n] for n in names]
    n = len(names)

    # Optimization objective: find best weights AND temperatures
    def neg_map3(params):
        weights = np.abs(params[:n])
        weights /= weights.sum()
        temps = np.abs(params[n:]) + 0.1  # temperatures > 0.1

        blended = sum(w * apply_temperature(o, t)
                      for w, o, t in zip(weights, oofs, temps))
        top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in blended]
        return -map_at_3(y_all, top3)

    # Grid search initialization
    print("\nSearching for optimal ensemble weights...")
    best_score, best_params = -1, None

    # Quick coarse grid for weights (temperatures fixed at 1.0)
    from itertools import product
    weight_grid = np.arange(0, 1.05, 0.1)
    for combo in product(weight_grid, repeat=n):
        if abs(sum(combo) - 1.0) > 0.05 or sum(combo) == 0:
            continue
        params = list(combo) + [1.0] * n
        s = -neg_map3(params)
        if s > best_score:
            best_score, best_params = s, params

    # Fine-tune with scipy
    result = minimize(neg_map3, best_params, method='Nelder-Mead',
                      options={'maxiter': 2000, 'xatol': 1e-5})
    final_weights = np.abs(result.x[:n])
    final_weights /= final_weights.sum()
    final_temps = np.abs(result.x[n:]) + 0.1

    print(f"\nOptimized weights: {dict(zip(names, final_weights.round(4)))}")
    print(f"Optimized temps:   {dict(zip(names, final_temps.round(4)))}")
    print(f"Ensemble OOF MAP@3: {-result.fun:.4f}")

    # Apply to test predictions
    test_names = [n for n in names if n in model_test]
    ensemble_test = sum(
        final_weights[names.index(n)] * apply_temperature(model_test[n], final_temps[names.index(n)])
        for n in test_names
    )

    # Also check if any model WITHOUT OOF has test predictions
    for name in model_test:
        if name not in model_oof:
            # Give it a reasonable weight
            print(f"  Adding {name} with heuristic weight 0.3")
            ensemble_test = 0.7 * ensemble_test + 0.3 * model_test[name]

elif len(model_test) == 1:
    name = list(model_test.keys())[0]
    ensemble_test = model_test[name]
    print(f"Single model: {name}")
else:
    # Multiple test models but limited OOF — use heuristic weights
    print("\nUsing heuristic weights (limited OOF data)...")
    # LLM gets highest weight, then LGBM, then TextCNN
    heuristic_weights = {"llm": 0.65, "lgbm": 0.20, "textcnn": 0.15}
    ensemble_test = sum(
        heuristic_weights.get(n, 0.1) * model_test[n]
        for n in model_test
    )
    total_w = sum(heuristic_weights.get(n, 0.1) for n in model_test)
    ensemble_test /= total_w

# ── Generate final submission ──
top3 = [[id2label[i] for i in np.argsort(-p)[:3]] for p in ensemble_test]
sub = pd.DataFrame({"ID": test["id"], "Prediction": [" ".join(t) for t in top3]})
sub.to_csv("submission.csv", index=False)
print(f"\n{'='*50}")
print(f"FINAL submission.csv saved ({len(sub)} rows)")
print(f"{'='*50}")
print(sub.head(10))

# Validation checks
assert all(len(p.split()) == 3 for p in sub['Prediction']), "Format error: not all rows have 3 predictions!"
assert all(all(x in LABELS for x in p.split()) for p in sub['Prediction']), "Format error: invalid labels!"
print("\n✓ All format checks passed!")
print("✓ Ready to submit to Kaggle!")


  ✓ lgbm: OOF MAP@3 = 1.0000
  ✓ textcnn: OOF MAP@3 = 0.9954
  ✓ llm: OOF MAP@3 = 0.8950

Models with OOF: ['lgbm', 'textcnn', 'llm']
Models with test: ['lgbm', 'textcnn', 'llm']

Searching for optimal ensemble weights...

Optimized weights: {'lgbm': np.float64(0.2), 'textcnn': np.float64(0.6), 'llm': np.float64(0.2)}
Optimized temps:   {'lgbm': np.float64(1.1), 'textcnn': np.float64(1.1), 'llm': np.float64(1.1)}
Ensemble OOF MAP@3: 1.0000

FINAL submission.csv saved (500 rows)
   ID Prediction
0   1      A B E
1   2      B A D
2   3      B C E
3   4      E C A
4   5      C D B
5   6      D A C
6   7      E A C
7   8      B E A
8   9      C A D
9  10      B C D

✓ All format checks passed!
✓ Ready to submit to Kaggle!
